# Link2iOS — The Legend of Zelda: Breath of the Wild

Pipeline dedicado BOTW. Entrada: Drive com `base/`, `update/`, `dlc/` e `prod.keys`. O processamento Linux acontece no Colab; a IPA Route C é compilada no GitHub Actions/macOS. RomFS e AOC são publicados separadamente no Drive e nunca entram na IPA.

**Antes de executar:** compartilhe a pasta de entrada com a service account como Editor e crie o Secret do Colab `LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON`.

In [ ]:
DRIVE_FOLDER_ID = input("ID da pasta BOTW no Google Drive: ").strip()
assert DRIVE_FOLDER_ID, "Informe o ID da pasta compartilhada."
from google.colab import userdata
assert userdata.get("LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON"), "Crie o Secret LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON no Colab."
print("Pasta de entrada configurada; credencial de service account encontrada.")


## 1. Obter o pipeline desta branch

Este clone é somente para ler os scripts da branch `Link2iOS`; nenhuma alteração local é enviada automaticamente.

In [ ]:
%cd /content
!rm -rf /content/Link2iOS-repo
!git clone --depth 1 --branch Link2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/Link2iOS-repo
!git -C /content/Link2iOS-repo rev-parse --short HEAD


## 2. Colab/Linux: baixar NSPs, preparar módulos/shaders e publicar os dados no Drive

A pasta de entrada precisa conter `base/*.nsp`, `update/*.nsp`, `dlc/*.nsp` e `prod.keys`. O script verifica os metadados dos pacotes; o nome dos arquivos não determina qual é base/update/DLC. O profile aceito é BOTW 1.9.0 (`v1114112`).

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py prepare --drive-folder-id "$DRIVE_FOLDER_ID"


## 3. Última célula: enviar o handoff e disparar somente Link2iOS

A última célula pede seu Fine-grained PAT. Ele precisa estar limitado a este repositório com **Contents: Read and write**. O token é usado apenas para criar/atualizar `handoff.json` na branch `Link2iOS`; ele não é salvo no notebook, Drive ou repositório. Esse commit é o gatilho do único workflow da branch.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py dispatch --handoff /content/link2ios-handoff.json
